# Indie Comic Pipeline - Google Colab Edition
This notebook runs the Indie Comic Pipeline with free GPU acceleration for SDXL (5-10 seconds per image instead of 2 hours on CPU).

### How to run:
1. In the Colab menu, go to **Runtime > Change runtime type** and select **T4 GPU** (or any available GPU).
2. Upload your `indie_comic_pipeline` folder to this session (or mount your Google Drive).
3. Run the cells below in order.

### Step 1: Install Dependencies
This installs the required libraries including PyTorch with GPU compatibility, diffusers, accelerate, and langchain.

In [ ]:
!pip install diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python pillow

### Step 2: Install and Start Ollama
This downloads the Ollama binary, launches the background daemon inside Colab, and pulls the `llama3.2` model.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama serve in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Give it 5 seconds to wake up

# Pull Llama 3.2 model
!ollama pull llama3.2

### Step 3: Configure Settings for Colab GPU
We will rewrite/update `config/settings.yaml` to use `cuda` (GPU device) and ensure standard SDXL resolution.

In [ ]:
import yaml

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for fast GPU SDXL
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['generation']['default_size']['width'] = 1024
settings['generation']['default_size']['height'] = 1024
settings['generation']['inference_steps'] = 30

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)

### Step 4: Run LangChain Extraction Pipeline
Change the arguments below to customize the character and story setting.

In [ ]:
# Customize character and based story setting here:
CHARACTER = "Spiderman"
STORY_SETTING = "Cyberpunk"

!python langchain_code/character_extractor.py "{CHARACTER}"
!python langchain_code/story_extractor.py "{STORY_SETTING}"
!python langchain_code/fusion_engine.py

### Step 5: Run SDXL Image Generation Pipeline
This loads SDXL on GPU, renders the character reference and the 4 story components, and saves them.

In [ ]:
%cd sdxl_code
!python run_sdxl_pipeline.py
%cd ..

### Step 5: Read the 10-Page Multiverse Crossover Script
This displays the full 10-page crossover storyboard script, including plot progression, character personality adaptation, panel descriptions, and dialogue/captions.

In [ ]:
import json

with open('outputs/fusion/fusion_complete.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

fusion = data.get('fusion', {})
personality = data.get('personality', {})
setting = data.get('setting', {})

print("=" * 80)
print(f"MULTIVERSE CROSSOVER EVENT: {personality.get('character_name')} IN {setting.get('story_name')}")
print(f"Theme of the World: {setting.get('theme', '')}")
print(f"Dialogue Style & Tone: {setting.get('dialogue_style_and_tone', '')}")
print(f"Cinematographic Visual Style: {setting.get('cinematographic_visual_style', '')}")
print(f"Adaptation Style: {fusion.get('character_visual_looks', '')}")
print(f"Adaptation Summary: {fusion.get('multiverse_adaptation_summary', '')}")
print("=" * 80)

pages = fusion.get('storyboard_10_pages', [])
for p in pages:
    print(f"\n=== PAGE {p.get('page_number')}: {p.get('location')} ===")
    print(f"Narrative Progression: {p.get('narrative_progression')}")
    print(f"Personality/Mood Shift: {p.get('personality_state')}")
    if p.get('side_characters_present'):
        print(f"Side Characters: {', '.join(p.get('side_characters_present'))}")
    print("Panels Breakdown:")
    for pb in p.get('panels_breakdown', []):
        print(f"  - {pb}")
    print("Dialogue & Captions:")
    for dc in p.get('dialogue_and_captions', []):
        print(f"  - {dc}")
    print("-" * 80)

### Step 6: View the Generated Assets (Visual Components)
Displays the generated character reference profile and the compiled dynamic asset sheet showing the persistent components (ranked by vector space persistence scoring).

In [ ]:
from IPython.display import Image, display

print("=== Character Reference Profile ===")
display(Image("outputs/characters/character_reference.png"))

print("\n=== Persistent Visual Assets Sheet (Dynamic Grid) ===")
display(Image("outputs/comics/component_sheet_grid_2x2.png"))